# End-to-End Generative AI for Business Report Summarization using RAG

**Project:** Infosys ESG Report 2024–25

**Pipeline:** PDF → text extraction → cleaning → chunking → embeddings → retrieval → Generative AI → Executive + Analytical summaries → Streamlit app


## 1. Install required packages

Run this once. The Transformers version is pinned to avoid the loading problem encountered earlier.

In [ ]:
!pip install -q pypdf sentence-transformers scikit-learn "transformers==4.57.1" sentencepiece torch streamlit

## 2. Check the project PDF

In [ ]:
from pathlib import Path

PDF_PATH = "infosys_esg_report.pdf"

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(
        f"{PDF_PATH} was not found. Put the Infosys ESG Report 2024-25 PDF "
        "in the same folder as this notebook."
    )

print("PDF found:", PDF_PATH)

## 3. Extract text from the PDF

In [ ]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(raw_text))
print("\nFirst 2000 characters:\n")
print(raw_text[:2000])

## 4. Clean and preprocess the text

In [ ]:
import re

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

clean_text_data = clean_text(raw_text)

print("Original characters:", len(raw_text))
print("Cleaned characters:", len(clean_text_data))
print(clean_text_data[:2000])

## 5. Split the report into chunks

In [ ]:
def chunk_text(text, chunk_size=1800, overlap=300):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

chunks = chunk_text(clean_text_data)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

## 6. Create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", chunk_embeddings.shape)

## 7. Build the retrieval layer

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "chunk": chunks[i],
            "score": float(scores[i]),
            "index": int(i)
        }
        for i in top_indices
    ]

test_query = "What are Infosys's key environmental goals?"
retrieved = retrieve(test_query, top_k=3)

for i, item in enumerate(retrieved, 1):
    print(f"--- Retrieved chunk {i} | similarity={item['score']:.3f} ---")
    print(item["chunk"][:1200])
    print()

## 8. Load the Generative AI model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Generative AI model loaded successfully.")

## 9. Connect Retrieval + Generation

In [ ]:
def generate_from_context(instruction, context, max_new_tokens=250):
    prompt = (
        instruction
        + "\n\nUse only the information in the context below. "
          "Do not invent facts or numbers.\n\n"
        + "Context:\n"
        + context
        + "\n\nAnswer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_answer(question, top_k=5):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n\n".join(item["chunk"] for item in retrieved)
    answer = generate_from_context(
        "Answer the question clearly and concisely.",
        context
    )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

result = rag_answer("What are Infosys's key environmental goals?", top_k=5)

print("Question:", result["question"])
print("\nRAG Answer:\n")
print(result["answer"])

## 10. Generate the Executive Summary

In [ ]:
executive_questions = [
    "What is the overall purpose and strategic direction of this report?",
    "What are the most important environmental goals, achievements, and risks?",
    "What are the most important social goals, achievements, and risks?",
    "What are the most important governance priorities and risks?",
    "What are the most important measurable results or KPIs reported?"
]

executive_parts = []

for question in executive_questions:
    result = rag_answer(question, top_k=4)
    executive_parts.append(
        "Question: " + question + "\nAnswer: " + result["answer"]
    )

executive_evidence = "\n\n".join(executive_parts)

executive_summary = generate_from_context(
    "Create a professional executive summary for CXOs. "
    "Include overall direction, key achievements, risks, strategic priorities, "
    "and important metrics.",
    executive_evidence,
    max_new_tokens=450
)

print(executive_summary)

## 11. Generate the Analytical Summary

In [ ]:
analytical_questions = [
    "List important environmental metrics, targets, and achievements with numbers.",
    "List important social metrics, targets, and achievements with numbers.",
    "List important governance, privacy, cybersecurity, and compliance metrics.",
    "What trends or gaps can be identified from the reported data?",
    "What are the most important risks or areas requiring management attention?"
]

analytical_parts = []

for question in analytical_questions:
    result = rag_answer(question, top_k=5)
    analytical_parts.append(
        "Question: " + question + "\nEvidence-based answer: " + result["answer"]
    )

analytical_evidence = "\n\n".join(analytical_parts)

analytical_summary = generate_from_context(
    "Create a professional analytical summary using bullet-style points. "
    "Preserve reported numbers and targets accurately. Include metrics, trends, "
    "achievements, risks and implications. Do not invent facts.",
    analytical_evidence,
    max_new_tokens=500
)

print(analytical_summary)

## 12. Save the generated outputs

In [ ]:
with open("executive_summary.txt", "w", encoding="utf-8") as f:
    f.write(executive_summary)

with open("analytical_summary.txt", "w", encoding="utf-8") as f:
    f.write(analytical_summary)

print("Saved executive_summary.txt")
print("Saved analytical_summary.txt")

## 13. Create the Streamlit deployment file

In [ ]:
app_code = '\nimport streamlit as st\nfrom pypdf import PdfReader\nimport re\nimport numpy as np\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM\n\nst.set_page_config(page_title="Business Report AI Summarizer", layout="wide")\nst.title("Business Report AI Summarizer")\nst.write("Upload a business report to generate an executive and analytical summary.")\n\n@st.cache_resource\ndef load_models():\n    embedder = SentenceTransformer("all-MiniLM-L6-v2")\n    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")\n    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")\n    return embedder, tokenizer, model\n\ndef clean_text(text):\n    text = text.replace("\\x00", " ")\n    text = re.sub(r"\\r\\n?", "\\n", text)\n    text = re.sub(r"[ \\t]+", " ", text)\n    text = re.sub(r"\\n{3,}", "\\n\\n", text)\n    return text.strip()\n\ndef chunk_text(text, chunk_size=1800, overlap=300):\n    chunks = []\n    start = 0\n    while start < len(text):\n        end = min(start + chunk_size, len(text))\n        chunk = text[start:end].strip()\n        if chunk:\n            chunks.append(chunk)\n        if end >= len(text):\n            break\n        start = end - overlap\n    return chunks\n\ndef generate(tokenizer, model, instruction, context, max_new_tokens=350):\n    prompt = (\n        instruction\n        + "\\n\\nUse only the information in the context below. "\n          "Do not invent facts or numbers.\\n\\n"\n        + "Context:\\n" + context + "\\n\\nAnswer:"\n    )\n    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)\n    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    return tokenizer.decode(outputs[0], skip_special_tokens=True)\n\nuploaded = st.file_uploader("Upload a PDF business report", type=["pdf"])\n\nif uploaded:\n    reader = PdfReader(uploaded)\n    text = "\\n".join(page.extract_text() or "" for page in reader.pages)\n    text = clean_text(text)\n    chunks = chunk_text(text)\n\n    embedder, tokenizer, model = load_models()\n    embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=False)\n\n    def retrieve(query, k=5):\n        q = embedder.encode([query], normalize_embeddings=True)\n        scores = cosine_similarity(q, embeddings)[0]\n        idx = np.argsort(scores)[::-1][:k]\n        return [chunks[i] for i in idx]\n\n    with st.spinner("Generating summaries..."):\n        questions = [\n            "What is the overall purpose and strategic direction of this report?",\n            "What are the most important environmental achievements, targets and risks?",\n            "What are the most important social achievements, targets and risks?",\n            "What are the most important governance priorities and risks?",\n            "What are the most important measurable KPIs?"\n        ]\n\n        evidence = []\n        for q in questions:\n            context = "\\n\\n".join(retrieve(q))\n            evidence.append("Question: " + q + "\\nContext: " + context)\n\n        all_evidence = "\\n\\n".join(evidence)\n\n        executive = generate(\n            tokenizer, model,\n            "Write a concise executive summary for CXOs. Include strategic direction, "\n            "achievements, risks, priorities and metrics.",\n            all_evidence,\n            450\n        )\n\n        analytical = generate(\n            tokenizer, model,\n            "Create a bullet-style analytical summary. Preserve numbers and targets. "\n            "Include metrics, trends, achievements, risks and implications.",\n            all_evidence,\n            500\n        )\n\n    st.subheader("Executive Summary")\n    st.write(executive)\n\n    st.subheader("Analytical Summary")\n    st.write(analytical)\n'

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('Created app.py')
print('Launch with: streamlit run app.py')

## 14. Conclusion

The completed project demonstrates:

- PDF ingestion and text extraction
- Text cleaning and preprocessing
- Chunking with overlap
- Embedding generation
- Similarity-based retrieval
- Retrieval-Augmented Generation
- Executive summarization
- Analytical, data-backed summarization
- Streamlit deployment

Keep the original Infosys ESG Report PDF in the same folder as this notebook when running it.
